# Receipt classification debugger

This notebook reproduces the worker's image path without RabbitMQ:

1. show the source image;
2. call the same Docling extractor method as the worker;
3. classify the extracted text with the production classification service; and
4. compare OCR-based receipt parsing with vision-based receipt parsing.

It captures the exact LLM prompt and raw response. Run cells in order. The cells that call Anthropic incur API usage.

In [ ]:
from __future__ import annotations

import json
import logging
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display
from PIL import Image

# Run Jupyter from python-worker/. If it starts elsewhere, find that directory.
project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = next(
        parent for parent in Path.cwd().resolve().parents if (parent / 'src').is_dir()
    )
os.chdir(project_root)

load_dotenv(project_root / '.env', override=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%Y-%m-%dT%H:%M:%S',
    force=True,
)

print(f'Using worker project: {project_root}')

In [ ]:
# Override this with RECEIPT_DEBUG_IMAGE=/path/to/your/receipt.jpg when needed.
image_path = Path(
    os.getenv(
        'RECEIPT_DEBUG_IMAGE',
        '/Users/thomas/Downloads/receipts/P0 (2).jpg',
    )
).expanduser().resolve()

if not image_path.is_file():
    raise FileNotFoundError(
        f'Image not found: {image_path}. Set RECEIPT_DEBUG_IMAGE and rerun this cell.'
    )

print(f'Image: {image_path}')
print(f'Size: {image_path.stat().st_size / 1024:.1f} KiB')

In [ ]:
# Inspect the original before interpreting OCR output.
with Image.open(image_path) as source_image:
    print(f'Pixel size: {source_image.size}; mode: {source_image.mode}')
    display(source_image.copy())

In [ ]:
from src.extractors.docling_extractor import DoclingExtractor

# Use the exact adapter and extract() call used by IngestionJobProcessor.
# This returns Docling TextItems joined in document order after conversion.
extractor = DoclingExtractor(
    artifacts_path=os.getenv('DOCLING_ARTIFACTS_PATH') or None,
)
markdown = extractor.extract(str(image_path))

display(Markdown('## Joined Docling TextItems sent to the worker classifier'))
print(f'Extracted {len(markdown):,} characters across {markdown.count(chr(10)) + 1} lines.')
print(markdown)

In [ ]:
# Plain text is easier to copy into an issue or compare line-by-line.
print(markdown)

In [ ]:
import anthropic

from src.services.classification_service import ClassificationService
from src.services.receipt_parser import ReceiptParser
from src.services.utils import get_text_from_response


class CapturingMessages:
    """Proxy that records the exact request/response used by worker services."""

    def __init__(self, messages):
        self._messages = messages
        self.last_request = None
        self.last_response = None

    def create(self, **kwargs):
        self.last_request = kwargs
        self.last_response = self._messages.create(**kwargs)
        return self.last_response


class CapturingAnthropicClient:
    def __init__(self, client):
        self.messages = CapturingMessages(client.messages)


api_key = os.getenv('ANTHROPIC_API_KEY')
if not api_key:
    raise RuntimeError('ANTHROPIC_API_KEY is required for classification and parsing.')

client = anthropic.Anthropic(
    api_key=api_key,
    base_url=os.getenv('ANTHROPIC_BASE_URL') or None,
)
capturing_client = CapturingAnthropicClient(client)
classifier = ClassificationService(capturing_client)
receipt_parser = ReceiptParser(capturing_client)

In [ ]:
# Production classification path. This makes one LLM request.
classification = classifier.classify(markdown)
classification_raw = get_text_from_response(capturing_client.messages.last_response)
classification_prompt = capturing_client.messages.last_request['messages'][0]['content']

display(Markdown('## Classification result'))
display(JSON(classification))

display(Markdown('## Exact prompt sent to the classifier'))
print(classification_prompt)

display(Markdown('## Raw classifier response'))
print(classification_raw)

## Compare receipt parsing paths

Run both cells below even if classification says `document`. The OCR parser exposes whether extracted text is sufficient; the vision parser tells you whether the image itself is recognizable as a receipt. Each cell makes an LLM request.

In [ ]:
# OCR-text receipt parsing. This makes one LLM request.
ocr_receipt = receipt_parser.parse(markdown)
ocr_receipt_raw = get_text_from_response(capturing_client.messages.last_response)

display(Markdown('## OCR-text receipt parse'))
display(JSON(ocr_receipt))

display(Markdown('## Raw OCR-parser response'))
print(ocr_receipt_raw)

In [ ]:
# Vision receipt parsing. This makes one LLM request and sends a JPEG rendition of the image.
vision_receipt = receipt_parser.parse_with_vision(str(image_path))
vision_receipt_raw = get_text_from_response(capturing_client.messages.last_response)

display(Markdown('## Vision receipt parse'))
display(JSON(vision_receipt))

display(Markdown('## Raw vision-parser response'))
print(vision_receipt_raw)

In [ ]:
# Compact comparison for an issue or pull request.
debug_summary = {
    'image': str(image_path),
    'classification': classification,
    'ocr_parse_confidence': ocr_receipt.get('confidence'),
    'vision_parse_confidence': vision_receipt.get('confidence'),
    'ocr_merchant': ocr_receipt.get('merchant'),
    'vision_merchant': vision_receipt.get('merchant'),
    'ocr_total': ocr_receipt.get('total'),
    'vision_total': vision_receipt.get('total'),
}
display(JSON(debug_summary))

if classification.get('classification') != 'receipt' and vision_receipt.get('confidence', 0) >= 0.7:
    print('Signal: vision recognizes a likely receipt, but OCR-text classification does not.')
    print('Next step: improve the classification prompt or add an image-aware classification fallback.')
elif ocr_receipt.get('confidence', 0) < 0.7:
    print('Signal: OCR text is probably incomplete or inaccurate; inspect the Markdown above.')
else:
    print('Signal: inspect the original image and classifier prompt for receipt-specific cues.')